In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
import math, os, glob
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import collections

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt          
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
import spconv.pytorch as spconv
import pickle
from pickle import UnpicklingError
from torch.amp import GradScaler, autocast

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def _make_linear(in_ch: int, out_ch: int, bn: bool = True, relu: bool = True) -> nn.Sequential:
    layers: List[nn.Module] = [nn.Linear(in_ch, out_ch, bias=not bn)]
    if bn:
        layers.append(nn.BatchNorm1d(out_ch))
    if relu:
        layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

In [ ]:
TRAIN_CSV_DIR  = r"F:\\Aditya\\Lidar Semantic Segmentation\\AHN3 Tiles\\Uncertainty Added"    

NUM_CLASSES    = 4
LOCAL_SIZE     = 25.6     
CONTEXT_SIZE   = 128.0    

MAX_LOCAL_PTS  = 8192     
MAX_CTX_PTS    = 131072   
STRIDE_RATIO   = 0.5     

IMAGE_SIZE     = (128, 128)  
RESOLUTION     = 1.0          

VOXEL_SIZE     = 0.5         
GRID_SIZE      = (32, 64, 64) 
PRETRAINED_2D  = True

NUM_EPOCHS     = 30
BATCH_SIZE     = 8
INIT_LR        = 5e-4
WEIGHT_DECAY   = 1e-4
LAMBDA_SCC     = 0.5
WORKERS        = 0            
SAVE_DIR       = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print("AHN3 Config loaded.")

In [ ]:
import itertools

all_csv_files = sorted(glob.glob(os.path.join(TRAIN_CSV_DIR, "*.csv")))

if not all_csv_files:
    raise RuntimeError(f"No CSV files found in {TRAIN_CSV_DIR}")

random.seed(42)
shuffled  = all_csv_files.copy()
random.shuffle(shuffled)

split_idx             = int(len(shuffled) * 0.8)
train_csv_files_multi = shuffled[:split_idx]
val_csv_files_multi   = shuffled[split_idx:]

print(f"Total tiles : {len(all_csv_files)}")
print(f"Train tiles : {len(train_csv_files_multi)}")
print(f"Val   tiles : {len(val_csv_files_multi)}")

IN_POINT_FEAT = 13

assert IMAGE_SIZE[0] >= 64 and IMAGE_SIZE[1] >= 64
print(f"IN_POINT_FEAT set to {IN_POINT_FEAT}")

In [ ]:
from importlib.resources import path


class ALSPointCloudDataset(Dataset):
    """
    Multi-tile ALS point-cloud dataset with three normalisation fixes:
    """

    REQUIRED_COLS = {"x", "y", "z", "label"}

    def __init__(
        self,
        csv_paths,
        local_size:   float = 25.6,
        context_size: float = 128.0,
        max_local:    int   = 8192,
        max_context:  int   = 32768,
        stride_ratio: float = 0.5,
        augment:      bool  = False,
    ):
        super().__init__()
        if isinstance(csv_paths, (str, Path)):
            csv_paths = sorted(glob.glob(str(csv_paths)))
        self.csv_paths    = [str(p) for p in csv_paths]
        self.local_size   = local_size
        self.context_size = context_size
        self.max_local    = max_local
        self.max_context  = max_context
        self.stride_ratio = stride_ratio
        self.augment      = augment
        self._rng         = np.random.default_rng(0)

        self._clouds:   List[np.ndarray] = []
        self._z_floors: List[float]      = []   # Fix 1: per-tile ground floor
        self._windows:  List[Tuple[int, float, float]] = []

        for fi, path in enumerate(self.csv_paths):
            pts, z_floor = self._load_csv(path)
            self._clouds.append(pts)
            self._z_floors.append(z_floor)
            self._windows.extend(self._compute_windows(pts, fi))

        print(f"Dataset  : {len(self._windows)} windows from "
              f"{len(self.csv_paths)} file(s)")
        if len(self._z_floors) > 1:
            print(f"z_floors : min={min(self._z_floors):.1f}m  "
                  f"max={max(self._z_floors):.1f}m  "
                  f"(span={max(self._z_floors)-min(self._z_floors):.1f}m)")

    # ── CSV loading ────────────────────────────────────────────────────────────
    def _load_csv(self, path: str) -> Tuple[np.ndarray, float]:
        import pandas as pd
        df = pd.read_csv(path)
        
        # Identify feature columns (excluding label)
        cols = [c for c in df.columns if c != 'label']
        
        # REQUIRED: Ensure x, y, z are indices 0, 1, 2
        for c in ['z', 'y', 'x']:
            if c in cols:
                cols.remove(c)
                cols.insert(0, c)
                
        features = df[cols].values.astype(np.float32)
        labels = df['label'].values.astype(np.int64)
        
        # Store the absolute tile floor for the z_global calculation
        z_floor = float(df['z'].min())
        
        return np.hstack([features, labels.reshape(-1, 1)]), z_floor

    # ── sliding-window centres ─────────────────────────────────────────────────
    def _compute_windows(
        self, pts: np.ndarray, file_idx: int
    ) -> List[Tuple[int, float, float]]:
        stride = self.local_size * self.stride_ratio
        xs, ys = pts[:, 0], pts[:, 1]
        x_min, x_max = float(xs.min()), float(xs.max())
        y_min, y_max = float(ys.min()), float(ys.max())
        windows = []
        cx = x_min + self.local_size / 2.0
        while cx < x_max + self.local_size / 2.0:
            cy = y_min + self.local_size / 2.0
            while cy < y_max + self.local_size / 2.0:
                lm = (
                    (xs >= cx - self.local_size / 2) &
                    (xs <  cx + self.local_size / 2) &
                    (ys >= cy - self.local_size / 2) &
                    (ys <  cy + self.local_size / 2)
                )
                if lm.sum() >= 1:
                    windows.append((file_idx, cx, cy))
                cy += stride
            cx += stride
        return windows

    # ── sampling ──────────────────────────────────────────────────────────────
    @staticmethod
    def _sample(idx: np.ndarray, n: int, rng) -> np.ndarray:
        if len(idx) == 0:
            return np.zeros(n, dtype=np.int64)
        return rng.choice(idx, n, replace=len(idx) < n)

    # ── augmentation ──────────────────────────────────────────────────────────
    @staticmethod
    def _augment(pts: np.ndarray, rng) -> np.ndarray:
        pts = pts.copy()
        angle = rng.uniform(0, 2 * np.pi)
        c, s  = np.cos(angle), np.sin(angle)
        pts[:, :2] = pts[:, :2] @ np.array([[c, s], [-s, c]], dtype=np.float32)
        if rng.random() > 0.5: pts[:, 0] = -pts[:, 0]
        
        scale = rng.uniform(0.95, 1.05)
        pts[:, :3] *= scale
        
        jitter = rng.normal(0, 0.01, size=(pts.shape[0], 3))
        pts[:, :3] += jitter
        
        return pts

    # ── __getitem__ ────────────────────────────────────────────────────────────
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # Get window metadata: file index, and center coordinates
        fi, cx, cy = self._windows[idx]
        pts      = self._clouds[fi]          # (N, num_cols) including label
        z_floor  = self._z_floors[fi]        # Absolute min Z for this tile

        xs, ys   = pts[:, 0], pts[:, 1]
        hs       = self.local_size   / 2.0
        hc       = self.context_size / 2.0

        # Create masks for local patch and context window
        l_mask = ((xs >= cx-hs) & (xs < cx+hs) & (ys >= cy-hs) & (ys < cy+hs))
        c_mask = ((xs >= cx-hc) & (xs < cx+hc) & (ys >= cy-hc) & (ys < cy+hc))

        # Sample indices to fixed point counts
        l_idx = self._sample(np.where(l_mask)[0], self.max_local,   self._rng)
        c_idx = self._sample(np.where(c_mask)[0], self.max_context, self._rng)

        # Slice features (all columns except the last one, which is 'label')
        p_local   = pts[l_idx, :-1].copy()   
        p_context = pts[c_idx, :-1].copy()
        labels    = pts[l_idx, -1].astype(np.int64)

        # 1. Compute context window's spatial floor for relative normalization
        ctx_x_min = float(p_context[:, 0].min())
        ctx_y_min = float(p_context[:, 1].min())
        ctx_z_min = float(p_context[:, 2].min())   

        # 2. Shift XY: Context window starts at (0, 0) for correct BEV alignment
        p_local[:,   0] -= ctx_x_min
        p_local[:,   1] -= ctx_y_min
        p_context[:, 0] -= ctx_x_min
        p_context[:, 1] -= ctx_y_min

        # 3. Compute z_global (Fix 1): Feature 28
        # This uses the tile's absolute floor to keep elevation context
        z_global_local   = (p_local[:,   2] - z_floor).reshape(-1, 1)
        z_global_context = (p_context[:, 2] - z_floor).reshape(-1, 1)

        # 4. Normalize Z relative to context floor for consistent RHFL signal
        p_local[:,   2] -= ctx_z_min
        p_context[:, 2] -= ctx_z_min

        # 5. Concatenate z_global as the final (28th) channel
        p_local   = np.hstack([p_local,   z_global_local])
        p_context = np.hstack([p_context, z_global_context])

        # Apply spatial augmentations if enabled
        if self.augment:
            p_local   = self._augment(p_local,   self._rng)
            p_context = self._augment(p_context, self._rng)

        return {
            "p_local":   torch.from_numpy(p_local).float(),   # (max_local, 28)
            "p_context": torch.from_numpy(p_context).float(), # (max_context, 28)
            "labels":    torch.from_numpy(labels).long(),     # (max_local,)
        }

    def __len__(self) -> int:
        return len(self._windows)


In [ ]:
def pce_collate(batch: List[Dict]) -> Dict:
    p_locals, p_ctxs, lbls = [], [], []
    bi_local_list, bi_ctx_list = [], []

    for i, sample in enumerate(batch):
        n = sample["p_local"].shape[0]
        m = sample["p_context"].shape[0]
        p_locals.append(sample["p_local"])
        p_ctxs.append(sample["p_context"])
        lbls.append(sample["labels"])
        bi_local_list.append(torch.full((n,), i, dtype=torch.long))
        bi_ctx_list.append(torch.full((m,), i, dtype=torch.long))

    return {
        "p_local":    torch.cat(p_locals, dim=0),
        "p_context":  torch.cat(p_ctxs,   dim=0),
        "labels":     torch.cat(lbls,      dim=0),
        "bi_local":   torch.cat(bi_local_list, dim=0),
        "bi_context": torch.cat(bi_ctx_list,   dim=0),
        "batch_size": len(batch),
    }

CACHE_PATH = "./cache/datasets.pkl"
os.makedirs("./cache", exist_ok=True)

loaded_from_cache = False
if os.path.exists(CACHE_PATH):
    print("Attempting to load datasets from cache")
    try:
        with open(CACHE_PATH, "rb") as f:
            cache = pickle.load(f)
        train_dataset = cache["train_dataset"]
        val_dataset   = cache["val_dataset"]
        print("Successfully loaded from cache.")
        loaded_from_cache = True
    except (UnpicklingError, EOFError, AttributeError) as e:
        print(f"Cache corrupted ({e}). Rebuilding...")

if not loaded_from_cache:
    print("Building datasets from CSVs")
    train_dataset = ALSPointCloudDataset(
        train_csv_files_multi,
        local_size=LOCAL_SIZE, context_size=CONTEXT_SIZE,
        max_local=MAX_LOCAL_PTS, max_context=MAX_CTX_PTS,
        stride_ratio=STRIDE_RATIO, augment=True,
    )
    val_dataset = ALSPointCloudDataset(
        val_csv_files_multi,
        local_size=LOCAL_SIZE, context_size=CONTEXT_SIZE,
        max_local=MAX_LOCAL_PTS, max_context=MAX_CTX_PTS,
        stride_ratio=STRIDE_RATIO, augment=False,
    )
    with open(CACHE_PATH, "wb") as f:
        pickle.dump({"train_dataset": train_dataset, "val_dataset": val_dataset}, f)
    print("Datasets built and cached.")

print(f"Train windows : {len(train_dataset)}")
print(f"Val   windows : {len(val_dataset)}")

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    collate_fn=pce_collate, 
    num_workers=WORKERS,   
    pin_memory=True,
    persistent_workers= WORKERS > 0, 
    prefetch_factor=2 if WORKERS > 0 else None     
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    collate_fn=pce_collate, 
    num_workers=WORKERS,   
    pin_memory=True,
    persistent_workers= WORKERS > 0, 
    prefetch_factor=2 if WORKERS > 0 else None
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

In [ ]:
class ContextProjection(nn.Module):
    """
    Projects P_context (M, 28) into a 28-channel BEV image.
    Aggregates all features (x, y, z, r, g, b, geometric features, etc.) 
    by averaging them within each grid cell.
    """
    def __init__(self, image_size: Tuple[int, int], resolution: float = 0.5, in_channels: int = 28):
        super().__init__()
        self.image_size = image_size
        self.resolution = resolution
        self.in_channels = in_channels

    def forward(self, p_context, batch_idx=None, batch_size=1):
        H, W = self.image_size
        r    = self.resolution
        C_in = self.in_channels
        
        if batch_idx is None:
            batch_idx = p_context.new_zeros(p_context.shape[0], dtype=torch.long)

        # Spatial coordinates for binning (assumed to be indices 0 and 1)
        x, y = p_context[:, 0], p_context[:, 1]
        u_all = torch.zeros_like(x, dtype=torch.long)
        v_all = torch.zeros_like(y, dtype=torch.long)

        # Calculate pixel coordinates per batch
        for b in range(batch_size):
            mask = batch_idx == b
            if not mask.any(): continue
            xb, yb = x[mask], y[mask]
            u_all[mask] = torch.floor((xb - xb.min()) / r).long().clamp(0, W - 1)
            v_all[mask] = torch.floor((yb - yb.min()) / r).long().clamp(0, H - 1)

        flat_idx  = batch_idx * (H * W) + v_all * W + u_all
        total_pix = batch_size * H * W

        # Aggregate ALL features into the BEV grid
        # Shape: (total_pix, C_in)
        sum_feats = p_context.new_zeros(total_pix, C_in).scatter_add_(
            0, flat_idx.unsqueeze(1).expand(-1, C_in), p_context
        )
        cnt = p_context.new_zeros(total_pix).scatter_add_(0, flat_idx, torch.ones_like(x))
        
        valid = cnt > 0
        avg_feats = torch.where(valid.unsqueeze(1), sum_feats / cnt.clamp(1).unsqueeze(1), torch.zeros_like(sum_feats))
        
        # Reshape to Image format: (B, C, H, W)
        image = avg_feats.view(batch_size, H, W, C_in).permute(0, 3, 1, 2).contiguous()

        # Metadata for the fusion branch
        pixel_coords = torch.stack([u_all, v_all], dim=1)            
        
        # Extract relative height (z is index 2) for the Disentangling block
        z_col = p_context[:, 2]
        min_z = p_context.new_full((total_pix,), 1e9)
        min_z.index_reduce_(0, flat_idx, z_col, reduce="amin", include_self=True)
        rel_height = z_col - min_z[flat_idx]
        
        return image, pixel_coords, rel_height

In [ ]:
class SPVConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, grid_size=(16, 16, 16)):
        super().__init__()
        self.grid_size = grid_size

        self.vox_conv = spconv.SparseSequential(
            spconv.SubMConv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False,
                              indice_key="subm0"),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.mlp_pre  = _make_linear(in_ch  * 2, out_ch)
        self.mlp_post = _make_linear(out_ch * 2, out_ch)

    def _voxelise(self, coords_int, feats, B):
        D, H, W = self.grid_size
        return spconv.SparseConvTensor(feats, coords_int, [D, H, W], B)

    def _interp(self, vox_dense, coords_norm, bi, B):
        C   = vox_dense.shape[1]
        out = vox_dense.new_zeros(coords_norm.shape[0], C, dtype=vox_dense.dtype)
        
        for b in range(B):
            m = bi == b
            if not m.any():
                continue
            g = coords_norm[m][:, [2, 1, 0]].view(1, m.sum(), 1, 1, 3).to(vox_dense.dtype)
            v = vox_dense[b].unsqueeze(0)
            
            o = F.grid_sample(v, g, mode="bilinear",
                              padding_mode="border", align_corners=True)
            
            out[m] = o.squeeze(0).squeeze(-1).squeeze(-1).T
        return out

    def forward(self, point_feat, point_xyz, bi, B):
        D, H, W = self.grid_size

        vc = torch.zeros_like(point_xyz)
        for b in range(B):
            m = bi == b
            if not m.any():
                continue
            pts_b   = point_xyz[m]
            xyz_min = pts_b.min(0).values
            span    = (pts_b.max(0).values - xyz_min).clamp(1e-6)
            vc[m]   = (pts_b - xyz_min) / span * point_xyz.new_tensor([D-1, H-1, W-1])

        ci = vc[:, 0].long().clamp(0, D-1)
        hi = vc[:, 1].long().clamp(0, H-1)
        wi = vc[:, 2].long().clamp(0, W-1)
        coords_int = torch.stack([bi, ci, hi, wi], dim=1).int()

        sp_in  = self._voxelise(coords_int, point_feat, B)
        sp_out = self.vox_conv(sp_in)

        vox_in_dense  = sp_in.dense()
        vox_out_dense = sp_out.dense()

        nc = vc.float().clone()
        nc[:, 0] = 2 * nc[:, 0] / max(D-1, 1) - 1
        nc[:, 1] = 2 * nc[:, 1] / max(H-1, 1) - 1
        nc[:, 2] = 2 * nc[:, 2] / max(W-1, 1) - 1

        pre_pts  = self._interp(vox_in_dense,  nc, bi, B)
        post_pts = self._interp(vox_out_dense, nc, bi, B)

        mlp_mid = self.mlp_pre(torch.cat([point_feat, pre_pts],  dim=1))
        return    self.mlp_post(torch.cat([mlp_mid,   post_pts], dim=1))

In [ ]:
class ResNet50FCN(nn.Module):
    """
    ResNet-50 backbone modified for 28-channel input with a 4-scale FCN decoder.
    """
    def __init__(self, in_channels=28, pretrained=True):
        super().__init__()
        # Load weights for the 3-channel version to transfer knowledge where possible
        weights  = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights)

        # Modify the first conv layer to accept 'in_channels'
        orig = backbone.conv1
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            backbone.bn1, 
            backbone.relu, 
            backbone.maxpool,
        )
        
        # Weight initialization for the new multi-channel input
        if pretrained:
            with torch.no_grad():
                # Average the pretrained RGB weights and repeat them across all 28 channels
                new_weight = orig.weight.data.mean(1, keepdim=True).repeat(1, in_channels, 1, 1)
                self.stem[0].weight.data = new_weight / (in_channels / 3.0)

        self.layer1 = backbone.layer1   # 256 channels
        self.layer2 = backbone.layer2   # 512 channels
        self.layer3 = backbone.layer3   # 1024 channels
        self.layer4 = backbone.layer4   # 2048 channels

        # Unified output dimension for the 2D/3D fusion
        OUT = 128
        self.proj1 = nn.Conv2d(256,  OUT, 1)
        self.proj2 = nn.Conv2d(512,  OUT, 1)
        self.proj3 = nn.Conv2d(1024, OUT, 1)
        self.proj4 = nn.Conv2d(2048, OUT, 1)
        self.out_channels = OUT

    def forward(self, img):
        H, W = img.shape[-2], img.shape[-1]
        
        # Downsampling path
        x1 = self.layer1(self.stem(img))   
        x2 = self.layer2(x1)               
        x3 = self.layer3(x2)               
        x4 = self.layer4(x3)               

        # Upsampling and projecting to a common feature dimension
        up = lambda t, p: F.interpolate(p(t), (H, W), mode="bilinear", align_corners=False)
        return [
            up(x1, self.proj1), 
            up(x2, self.proj2),
            up(x3, self.proj3), 
            up(x4, self.proj4)
        ]

    @staticmethod
    def query_context_features(feat_maps, pixel_coords, batch_idx):
        """
        Indices into the 2D feature maps using point-wise pixel coordinates.
        """
        out = []
        for feat in feat_maps:
            _, _, H, W = feat.shape
            u = pixel_coords[:, 0].clamp(0, W - 1)
            v = pixel_coords[:, 1].clamp(0, H - 1)
            # Use batch_idx to ensure points query the correct sample in the batch
            out.append(feat[batch_idx, :, v, u])
        return out

In [ ]:
class EmbeddingDisentangling(nn.Module):
    """RHFL + attention-based 2-D/3-D fusion (Eq. 5)."""
    def __init__(self, dim_3d, dim_2d, dim_out):
        super().__init__()
        self.height_proj = _make_linear(1, dim_2d)
        self.fuse_2d     = _make_linear(dim_2d * 2, dim_2d)
        self.fuse_3d     = _make_linear(dim_2d + dim_3d, dim_out)
        self.attn_gate   = nn.Linear(dim_out, dim_out)

    def forward(self, f3d, e2d, rel_h):
        h_feat  = self.height_proj(rel_h.unsqueeze(1))
        lifted  = self.fuse_2d(torch.cat([h_feat, e2d], dim=1))
        f_prime = self.fuse_3d(torch.cat([lifted, f3d], dim=1))
        return torch.sigmoid(self.attn_gate(f_prime)) * f_prime


In [ ]:
class _3DEncoder(nn.Module):
    DIMS = [32, 64, 128, 256]

    def __init__(self, in_ch, grid_size):
        super().__init__()
        d = self.DIMS
        self.b1 = SPVConvBlock(in_ch,  d[0], grid_size)
        self.b2 = SPVConvBlock(d[0],   d[1], grid_size)
        self.b3 = SPVConvBlock(d[1],   d[2], grid_size)
        self.b4 = SPVConvBlock(d[2],   d[3], grid_size)
        self.d1 = _make_linear(d[0], d[0])
        self.d2 = _make_linear(d[1], d[1])
        self.d3 = _make_linear(d[2], d[2])

    def forward(self, feat, xyz, bi, B):
        f1 = self.b1(feat,        xyz, bi, B)
        f2 = self.b2(self.d1(f1), xyz, bi, B)
        f3 = self.b3(self.d2(f2), xyz, bi, B)
        f4 = self.b4(self.d3(f3), xyz, bi, B)
        return [f1, f2, f3, f4]


class PCENet(nn.Module):
    """
    Full PCE model.
    FIX: added image_size and resolution to __init__ (Cell 6 in original
         notebook passed them but the class didn't accept them → TypeError).
    FIX: forward now returns real logits, not torch.randn.
    """
    def __init__(
        self,
        num_classes:   int,
        in_point_feat: int   = 13,  # Updated default
        image_size:    Tuple[int, int] = (128, 128),
        resolution:    float = 1.0,
        grid_size:     Tuple[int, int, int] = (16, 16, 16),
        pretrained_2d: bool  = True,
        lambda_scc:    float = 0.5,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.lambda_scc  = lambda_scc

        # Pass in_point_feat to BOTH projection and encoders
        self.context_proj = ContextProjection(image_size, resolution, in_channels=in_point_feat)
        self.encoder_2d   = ResNet50FCN(in_channels=in_point_feat, pretrained=pretrained_2d)
        self.encoder_3d   = _3DEncoder(in_point_feat, grid_size)

        dim_2d   = self.encoder_2d.out_channels   # 128
        dims_3d  = _3DEncoder.DIMS                # [32,64,128,256]

        self.ed_blocks = nn.ModuleList([
            EmbeddingDisentangling(dims_3d[i], dim_2d, dims_3d[i])
            for i in range(4)
        ])

        total_dim = sum(dims_3d)   # 480
        self.classifier = nn.Sequential(
            _make_linear(total_dim, 256),
            nn.Dropout(0.5),          
            _make_linear(256, 128),  
            nn.Dropout(0.5),          # Added
            nn.Linear(128, num_classes),
        )
        self.ssc_head   = nn.Linear(dim_2d, num_classes)
        self.seg_loss   = nn.CrossEntropyLoss(ignore_index=-1)
        self.scc_loss   = nn.MultiLabelSoftMarginLoss()

    def _soft_pixel_labels(self, labels, pix_coords, bi, B, H, W):
        C    = self.num_classes
        soft = torch.zeros(B*H*W, C, device=labels.device)
        u    = pix_coords[:, 0].clamp(0, W-1)
        v    = pix_coords[:, 1].clamp(0, H-1)
        flat = bi*(H*W) + v*W + u
        ok   = (labels >= 0) & (labels < C)
        for c in range(C):
            m = ok & (labels == c)
            if m.any():
                soft[flat[m].unique(), c] = 1.0
        return soft

    def forward(self, p_local, p_context, bi_local, bi_context,
                batch_size, labels=None):
        B = batch_size
        img, pix_ctx,   _          = self.context_proj(p_context, bi_context, B)
        _,   pix_local, rel_height = self.context_proj(p_local,   bi_local,   B)

        E_2d  = self.encoder_2d(img)
        q_2d  = ResNet50FCN.query_context_features(E_2d, pix_local, bi_local)
        F_3d  = self.encoder_3d(p_local, p_local[:, :3], bi_local, B)   # FIX: real forward

        fused = [self.ed_blocks[i](F_3d[i], q_2d[i], rel_height)
                 for i in range(4)]
        logits = self.classifier(torch.cat(fused, dim=1))

        out = {"logits": logits}

        if labels is not None:
            H, W = img.shape[-2], img.shape[-1]
            loss_seg  = self.seg_loss(logits, labels)
            e1_flat   = E_2d[0].permute(0,2,3,1).reshape(-1, E_2d[0].shape[1])
            scc_lgt   = self.ssc_head(e1_flat)
            soft_lbl  = self._soft_pixel_labels(labels, pix_local, bi_local, B, H, W)
            loss_scc  = self.scc_loss(scc_lgt, soft_lbl)
            out["loss"]     = loss_seg + self.lambda_scc * loss_scc
            out["loss_seg"] = loss_seg
            out["loss_scc"] = loss_scc
        return out


In [ ]:
model = PCENet(
    num_classes   = NUM_CLASSES,
    in_point_feat = IN_POINT_FEAT,    # 5 — includes z_global
    image_size    = IMAGE_SIZE,
    resolution    = RESOLUTION,
    grid_size     = GRID_SIZE,
    pretrained_2d = PRETRAINED_2D,
    lambda_scc    = LAMBDA_SCC,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

model = PCENet(
    num_classes   = NUM_CLASSES,
    in_point_feat = IN_POINT_FEAT,    # 5 — includes z_global
    image_size    = IMAGE_SIZE,
    resolution    = RESOLUTION,
    grid_size     = GRID_SIZE,
    pretrained_2d = PRETRAINED_2D,
    lambda_scc    = LAMBDA_SCC,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Logits shape : {_out['logits'].shape}")
print(f"Total loss   : {_out['loss'].item():.4f}")
print(f"Seg  loss    : {_out['loss_seg'].item():.4f}")
print(f"SCC  loss    : {_out['loss_scc'].item():.4f}")
del _out
print("Multi-tile smoke test passed ")


In [ ]:
def compute_iou(preds, labels, num_classes):
    """Returns per-class IoU as a list, ignoring index -1."""
    ious = []
    for c in range(num_classes):
        pred_c  = (preds == c)
        label_c = (labels == c)
        valid   = (labels != -1)
        inter   = (pred_c & label_c & valid).sum().item()
        union   = (( pred_c | label_c) & valid).sum().item()
        ious.append(inter / union if union > 0 else float("nan"))
    return ious


def train_one_epoch(model, loader, optimizer, device, num_classes, scaler):
    model.train()
    tot, seg, scc = 0.0, 0.0, 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc="Train", leave=False)

    for batch in pbar:
        p_local   = batch["p_local"].to(device)
        p_context = batch["p_context"].to(device)
        bi_local  = batch["bi_local"].to(device)
        bi_ctx    = batch["bi_context"].to(device)
        labels    = batch["labels"].to(device)
        B         = batch["batch_size"]

        optimizer.zero_grad()

        with autocast(device_type='cuda', enabled=True):
            out = model(p_local, p_context, bi_local, bi_ctx,
                        batch_size=B, labels=labels)
            loss = out["loss"]

        scaler.scale(loss).backward()
        
        scaler.step(optimizer)
        scaler.update()

        tot += out["loss"].item()
        seg += out["loss_seg"].item()
        scc += out["loss_scc"].item()

        preds = out["logits"].argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

        pbar.set_postfix(loss=f'{out["loss"].item():.3f}',
                         seg=f'{out["loss_seg"].item():.3f}')

    n           = len(loader)
    all_preds   = torch.cat(all_preds)
    all_labels  = torch.cat(all_labels)
    iou_per_cls = compute_iou(all_preds, all_labels, num_classes)
    miou        = np.nanmean(iou_per_cls)
    acc         = (all_preds[all_labels != -1] == all_labels[all_labels != -1]).float().mean().item()

    return {
        "loss": tot/n, "seg": seg/n, "scc": scc/n,
        "miou": miou,  "acc": acc,   "iou_per_cls": iou_per_cls,
    }


@torch.no_grad()
def validate(model, loader, device, num_classes):
    model.eval()
    tot, seg, scc = 0.0, 0.0, 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc="Val  ", leave=False)

    for batch in pbar:
        p_local   = batch["p_local"].to(device)
        p_context = batch["p_context"].to(device)
        bi_local  = batch["bi_local"].to(device)
        bi_ctx    = batch["bi_context"].to(device)
        labels    = batch["labels"].to(device)
        B         = batch["batch_size"]

        out = model(p_local, p_context, bi_local, bi_ctx,
                    batch_size=B, labels=labels)

        tot += out["loss"].item()
        seg += out["loss_seg"].item()
        scc += out["loss_scc"].item()

        preds = out["logits"].argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    n           = len(loader)
    all_preds   = torch.cat(all_preds)
    all_labels  = torch.cat(all_labels)
    iou_per_cls = compute_iou(all_preds, all_labels, num_classes)
    miou        = np.nanmean(iou_per_cls)
    acc         = (all_preds[all_labels != -1] == all_labels[all_labels != -1]).float().mean().item()

    return {
        "loss": tot/n, "seg": seg/n, "scc": scc/n,
        "miou": miou,  "acc": acc,   "iou_per_cls": iou_per_cls,
    }

In [ ]:
model = PCENet(
    num_classes   = NUM_CLASSES,
    in_point_feat = IN_POINT_FEAT,
    image_size    = IMAGE_SIZE,
    resolution    = RESOLUTION,
    grid_size     = GRID_SIZE,
    pretrained_2d = PRETRAINED_2D,
    lambda_scc    = LAMBDA_SCC,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=INIT_LR,           
    weight_decay=WEIGHT_DECAY   
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=2
)

scaler = GradScaler(device='cuda', enabled=True)

history = {
    "loss": [], "seg": [], "scc": [], "miou": [], "acc": [],
    "val_loss": [], "val_seg": [], "val_scc": [], "val_miou": [], "val_acc": [],
}

best_val_loss = float("inf")
print(f"Starting AMP training for {NUM_EPOCHS} epochs on {device} ...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer, device, NUM_CLASSES, scaler)
    
    vl = validate(model, val_loader, device, NUM_CLASSES)
    
    scheduler.step(vl["loss"])

    history["loss"].append(tr["loss"])
    history["seg"].append(tr["seg"])
    history["scc"].append(tr["scc"])
    history["miou"].append(tr["miou"])
    history["acc"].append(tr["acc"])

    history["val_loss"].append(vl["loss"])
    history["val_seg"].append(vl["seg"])
    history["val_scc"].append(vl["scc"])
    history["val_miou"].append(vl["miou"])
    history["val_acc"].append(vl["acc"])

    curr_lr = optimizer.param_groups[0]['lr']
    iou_str = "  ".join(f"C{c}={v:.3f}" for c, v in enumerate(vl["iou_per_cls"]))
    
    print(
        f"Epoch [{epoch:>3}/{NUM_EPOCHS}]  "
        f"Loss: {tr['loss']:.4f}  mIoU: {tr['miou']:.4f}  Acc: {tr['acc']:.4f}  |  "
        f"Val Loss: {vl['loss']:.4f}  Val mIoU: {vl['miou']:.4f}  Val Acc: {vl['acc']:.4f}\n"
        f"  Val IoU per class: {iou_str}  "
        f"LR: {curr_lr:.2e}"
    )

    if vl["loss"] < best_val_loss:
        best_val_loss = vl["loss"]
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "opt_state":   optimizer.state_dict(),
            "val_loss":    vl["loss"],
            "history":     history
        }, os.path.join(SAVE_DIR, "pce_best.pth"))
        print("  ↑ best model saved")

# Save final state
torch.save({
    "epoch":       NUM_EPOCHS,
    "model_state": model.state_dict(),
    "history":     history,
}, os.path.join(SAVE_DIR, "pce_final.pth"))

print("\nTraining complete. Checkpoints saved to", SAVE_DIR)

In [ ]:
epochs_x = range(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = [
    ("loss",  "val_loss",  "Total Loss"),
    ("seg",   "val_seg",   "Segmentation Loss"),
    ("scc",   "val_scc",   "Auxiliary SCC Loss"),
    ("miou",  "val_miou",  "Mean IoU"),
    ("acc",   "val_acc",   "Accuracy"),
]

for ax, (tr_key, vl_key, title) in zip(axes.flat, metrics):
    ax.plot(epochs_x, history[tr_key], label="Train", linewidth=2)
    ax.plot(epochs_x, history[vl_key], label="Val",   linewidth=2, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

# Per-class IoU bar chart in the last panel using final epoch val IoUs
ax_iou = axes.flat[5]
final_iou = history["val_iou_per_cls"][-1] if "val_iou_per_cls" in history else vl["iou_per_cls"]
classes   = [f"Class {c}" for c in range(NUM_CLASSES)]
bars = ax_iou.bar(classes, final_iou, color="steelblue")
ax_iou.set_title("Per-Class IoU (Final Epoch, Val)")
ax_iou.set_ylabel("IoU")
ax_iou.set_ylim(0, 1)
ax_iou.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, final_iou):
    if not np.isnan(val):
        ax_iou.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()